# ChakraModel v7 Verification (CPU-only, no GPU quota used)

Purpose: settle every open question before spending GPU hours.

1. Attach the code dataset (should be v7), the weights dataset, and your polyp datasets — same as the real run.
2. **Accelerator: set to "None" / CPU** — this notebook does not need a GPU. Saves your quota.
3. Run All. Paste the full output of the verification cell back.

This does NOT load any model and does NOT run evaluation — it only stages code and calls `dataset_discovery.discover()` for real, so what you see here is exactly what the real run would scan and score, without burning GPU time to find out.

In [1]:
import os, shutil, glob

WORK = '/kaggle/working'

code_zip = None
for p in glob.glob('/kaggle/input/**/ChakraModel_EvalHarness_Code*.zip', recursive=True):
    code_zip = p
    break

if code_zip:
    print('Unpacking code from', code_zip)
    shutil.unpack_archive(code_zip, WORK)
else:
    eh_dir = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if os.path.basename(root) == 'eval_harness' and 'run_eval.py' in files:
            eh_dir = root
            break
    assert eh_dir, (
        'Could not find ChakraModel_EvalHarness_Code*.zip or an extracted eval_harness/ '
        'folder anywhere under /kaggle/input — attach the code dataset.'
    )
    dataset_root = os.path.dirname(eh_dir)
    print('Found already-extracted code at', dataset_root)
    shutil.copytree(eh_dir, f'{WORK}/eval_harness', dirs_exist_ok=True)
    src_dir = os.path.join(dataset_root, 'src')
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, f'{WORK}/src', dirs_exist_ok=True)
    print('Copied eval_harness/ and src/ into', WORK)

os.makedirs(f'{WORK}/weights/checkpoints', exist_ok=True)
wanted = {'chakra_transformer_best.pth', 'combo1_best.pth'}
found = set()
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f in wanted and f not in found:
            src = os.path.join(root, f)
            dst = f'{WORK}/weights/checkpoints/{f}'
            shutil.copy(src, dst)
            found.add(f)
            print('Linked', f, 'from', src)
missing = wanted - found
if missing:
    print('WARNING missing weights:', missing)
else:
    print('Both checkpoints staged OK.')

Found already-extracted code at /kaggle/input/datasets/jayasrikannuchamy/chakramodel-evalharness-code-new
Copied eval_harness/ and src/ into /kaggle/working
Linked combo1_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/combo1_best.pth
Linked chakra_transformer_best.pth from /kaggle/input/datasets/jayasrikannuchamy/chakratransformer-weights/weights/checkpoints/chakra_transformer_best.pth
Both checkpoints staged OK.


In [2]:
!pip install -q opencv-python-headless

In [3]:
# ============================================================
# CPU-ONLY VERIFICATION — no model loading, no GPU needed.
# Answers: is this actually v7? what will discover() actually find?
# how many frames total? are weights present?
# ============================================================
import sys, os

WORK = '/kaggle/working'
sys.path.insert(0, f'{WORK}/eval_harness')
sys.path.insert(0, f'{WORK}/src')

print("="*70)
print("1. FILE VERSION CHECK")
print("="*70)

def check(path, must_contain, label):
    if not os.path.exists(path):
        print(f"  [MISSING] {path}")
        return
    src = open(path).read()
    ok = all(s in src for s in must_contain)
    print(f"  [{'OK' if ok else 'FAIL'}] {label}: {path}")
    for s in must_contain:
        print(f"       {'found' if s in src else 'MISSING'}: {s!r}")

check(f'{WORK}/eval_harness/resource_monitor.py',
      ['MAX_BATCH         = 8'], 'batch ceiling fix (v4+)')
check(f'{WORK}/eval_harness/eval_engine.py',
      ['mode="default"', '_key_renames', 'ppd_pred.weight'], 'compile-mode + checkpoint-remap fix (v4/v6+)')
check(f'{WORK}/eval_harness/dataset_discovery.py',
      ['_discover_recursive', 'MAX_FRAMES_PER_VIDEO', 'IterableDataset'], 'recursive discovery + video cap/sequential-decode fix (v5/v7)')

print()
print("="*70)
print("2. ACTUAL DATASET_DISCOVERY.PY BEHAVIOR (real scan, no GPU)")
print("="*70)
from dataset_discovery import discover
metas = discover('/kaggle/input')
total = 0
for m in metas:
    print(f"  {m.name:45s} | {m.kind.name:15s} | {m.frame_count:8d} frames")
    total += m.frame_count
print(f"\n  TOTAL FRAMES ACROSS ALL DATASETS: {total:,}")

print()
print("="*70)
print("3. WEIGHTS PRESENT?")
print("="*70)
for f in ['chakra_transformer_best.pth', 'combo1_best.pth']:
    p = f'{WORK}/weights/checkpoints/{f}'
    print(f"  {'OK' if os.path.exists(p) else 'MISSING'}: {p}")

1. FILE VERSION CHECK
  [OK] batch ceiling fix (v4+): /kaggle/working/eval_harness/resource_monitor.py
       found: 'MAX_BATCH         = 8'
  [OK] compile-mode + checkpoint-remap fix (v4/v6+): /kaggle/working/eval_harness/eval_engine.py
       found: 'mode="default"'
       found: '_key_renames'
       found: 'ppd_pred.weight'
  [OK] recursive discovery + video cap/sequential-decode fix (v5/v7): /kaggle/working/eval_harness/dataset_discovery.py
       found: '_discover_recursive'
       found: 'MAX_FRAMES_PER_VIDEO'
       found: 'IterableDataset'

2. ACTUAL DATASET_DISCOVERY.PY BEHAVIOR (real scan, no GPU)
  [DISCOVER] cvc-clinicdb                             | PAIRED_IMAGE    |    495 frames
  [DISCOVER] etis-larib                               | PAIRED_IMAGE    |      5 frames
  [DISCOVER] kvasir-seg                               | PAIRED_IMAGE    |    992 frames
  [DISCOVER] CVC-300                                  | PAIRED_IMAGE    |     60 frames
  [DISCOVER] hyper-kvasir-labele